In [2]:
from langchain_neo4j import Neo4jGraph
import os
from dotenv import load_dotenv
from configuration.config import *
load_dotenv()

True

In [5]:
graph=Neo4jGraph(
    url=NEO4J_CONFIG['uri'],
    username=NEO4J_CONFIG['auth'][0],
    password=NEO4J_CONFIG['auth'][1],
)

ValueError: Could not use APOC procedures. Please ensure the APOC plugin is installed in Neo4j and that 'apoc.meta.data()' is allowed in Neo4j configuration 

如何解决呢?neo4j官网的APOC说了,只需要将在labs下的apoc的jar包粘贴到plugins下然后重启neo4j就行了

```bash
nikofox@MOSS:/home/neo4j$ docker exec -it myneo4j /bin/bash
root@4c51457585cf:/var/lib/neo4j# ls
LICENSE.txt  README.txt  ThirdPartyLicenses.txt  UPGRADE.txt  bin  certificates  conf  data  import  labs  lib	licenses  logs	packaging_info	plugins  products  run	web
root@4c51457585cf:/var/lib/neo4j# cd labs/
root@4c51457585cf:/var/lib/neo4j/labs# ls
LICENSE  README.txt  apoc-2026.04.0-core.jar
root@4c51457585cf:/var/lib/neo4j/labs# cp ./apoc-2026.04.0-core.jar ../plugins/
root@4c51457585cf:/var/lib/neo4j/labs# cd ..
root@4c51457585cf:/var/lib/neo4j# cd plugins/
root@4c51457585cf:/var/lib/neo4j/plugins# ls
apoc-2026.04.0-core.jar
```

In [6]:
graph=Neo4jGraph(
    url=NEO4J_CONFIG['uri'],
    username=NEO4J_CONFIG['auth'][0],
    password=NEO4J_CONFIG['auth'][1],
)

ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureRegistrationFailed} {message: apoc.meta.data is unavailable because it is sandboxed and has dependencies outside of the sandbox. Sandboxing is controlled by the dbms.security.procedures.unrestricted setting. Only unrestrict procedures you can trust with access to database internals.} {gql_status: 52N34} {gql_status_description: error: procedure exception - procedure restricted. apoc.meta.data() is restricted and accesses database internals. Procedure restriction is controlled by the dbms.security.procedures.unrestricted setting. Only un-restrict procedures you can trust with access to database internals.}

你遇到的错误是 **Neo4j** 数据库在启用了 **沙箱（sandbox）安全机制** 的情况下，阻止了对 APOC 库中 `apoc.meta.data` 过程的调用。这个过程被归类为“受限过程（restricted procedure）”，因为它可以访问数据库内部结构，存在潜在安全风险。

---

### 🔒 错误核心信息解析：

- **错误代码**：`Neo.ClientError.Procedure.ProcedureRegistrationFailed`
- **关键提示**：
  > `apoc.meta.data is unavailable because it is sandboxed and has dependencies outside of the sandbox.`
- **解决方案提示**：
  > `Sandboxing is controlled by the dbms.security.procedures.unrestricted setting. Only unrestrict procedures you can trust.`

---

### ✅ 解决方法

你需要在 Neo4j 的配置文件中 **显式允许 `apoc.meta.data`（或整个 APOC 库）绕过沙箱限制**。

#### 步骤如下：

1. **打开 Neo4j 配置文件**
   通常是：
   - **Neo4j Desktop / Server**: `neo4j.conf`
   - 路径示例（Linux/macOS）: `/etc/neo4j/neo4j.conf`
   - Windows: `C:\Program Files\Neo4j\conf\neo4j.conf`

2. **添加或修改以下配置项**：

   ```properties
   # 允许特定 APOC 过程不受限（推荐方式）
   dbms.security.procedures.unrestricted=apoc.meta.data

   # 或者，如果你信任整个 APOC 库（更宽松，但需谨慎）：
   dbms.security.procedures.unrestricted=apoc.*
   ```

   > ⚠️ 注意：`dbms.security.procedures.unrestricted` 是一个逗号分隔的列表，可包含多个过程名。

3. **重启 Neo4j 服务**
   修改配置后必须重启才能生效：
   ```bash
   sudo systemctl restart neo4j      # Linux systemd
   # 或通过 Neo4j Desktop 重启实例
   ```

4. **验证是否生效**
   在 Neo4j Browser 中运行：
   ```cypher
   CALL apoc.meta.data()
   ```
   如果不再报错，说明配置成功。

---

### 🔐 安全建议

- **仅在可信环境中使用**：`apoc.meta.data()` 可以暴露数据库的 schema、标签、关系类型等元数据，**不建议在生产环境或开放网络中随意放开限制**。
- **最小权限原则**：如果只需要 `apoc.meta.data`，就只放行它，不要用 `apoc.*` 除非必要。
- **考虑替代方案**：如果你只是想获取图结构信息，也可以使用原生 Cypher 查询，例如：
  ```cypher
  CALL db.schema()
  ```
  这个命令不需要 APOC，且通常不受限。

---

### 📌 补充说明

- APOC（Awesome Procedures on Cypher）是一个强大的 Neo4j 扩展库。
- 自 Neo4j 4.x 起，出于安全考虑，默认启用了 **过程沙箱机制**，所有可能访问底层或执行高危操作的过程都被默认限制。
- `apoc.meta.data` 因其能读取内部元数据，被归入受限列表。

---

如你是在 **Docker、云服务（如 Neo4j Aura）或托管平台** 上运行 Neo4j，请注意：
- **Neo4j Aura 等 SaaS 服务通常不允许修改 `dbms.security.procedures.unrestricted`**，因此无法使用 `apoc.meta.data`。
- 此时应改用 `db.schema()` 或其他非受限过程。

如有具体部署环境（本地/Docker/Aura），可提供更多信息，我可以给出针对性建议。

```bash
root@MOSS:/home/neo4j# ls
conf  data  import  logs  plugins
root@MOSS:/home/neo4j# cd conf/
root@MOSS:/home/neo4j/conf# ls
neo4j.conf
root@MOSS:/home/neo4j/conf# vim neo4j.conf
root@MOSS:/home/neo4j/conf# cat neo4j.conf

server.memory.pagecache.size=512M

server.default_listen_address=0.0.0.0

db.temporal.timezone=Asia/Shanghai
dbms.db.timezone=SYSTEM
# 允许特定 APOC 过程不受限（推荐方式）  注意别跟上面重复定义,不然还是连不上
dbms.security.procedures.unrestricted=apoc.meta.data

server.directories.logs=/logs


```

In [9]:
graph=Neo4jGraph(
    url=NEO4J_CONFIG['uri'],
    username=NEO4J_CONFIG['auth'][0],
    password=NEO4J_CONFIG['auth'][1],
)

In [10]:
print(graph.schema)

Node properties:
Category1 {id: INTEGER, name: STRING}
Category2 {id: INTEGER, name: STRING}
Category3 {id: INTEGER, name: STRING}
BaseAttrName {id: INTEGER, name: STRING}
BaseAttrValue {id: INTEGER, name: STRING}
SPU {id: INTEGER, name: STRING}
SKU {id: INTEGER, name: STRING}
Trademark {id: INTEGER, name: STRING}
SaleAttrName {id: INTEGER, name: STRING}
SaleAttrValue {id: INTEGER, name: STRING}
Tag {id: STRING, name: STRING}
Relationship properties:

The relationships:
(:Category1)-[:Have]->(:BaseAttrName)
(:Category2)-[:Belong]->(:Category1)
(:Category2)-[:Have]->(:BaseAttrName)
(:Category3)-[:Have]->(:BaseAttrName)
(:Category3)-[:Belong]->(:Category2)
(:BaseAttrName)-[:Have]->(:BaseAttrValue)
(:SPU)-[:Have]->(:Tag)
(:SPU)-[:Have]->(:SaleAttrName)
(:SPU)-[:Belong]->(:Trademark)
(:SPU)-[:Belong]->(:Category3)
(:SKU)-[:Have]->(:BaseAttrValue)
(:SKU)-[:Have]->(:SaleAttrValue)
(:SKU)-[:Belong]->(:SPU)
(:SaleAttrName)-[:Have]->(:SaleAttrValue)


In [11]:
res=graph.query("""MATCH (n) RETURN n LIMIT 5""")
print(res)

[{'n': {'name': '图书、音像、电子书刊', 'id': 1}}, {'n': {'name': '手机', 'id': 2}}, {'n': {'name': '家用电器', 'id': 3}}, {'n': {'name': '数码', 'id': 4}}, {'n': {'name': '家居家装', 'id': 5}}]


定义大模型

In [18]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model_provider='openai',
    model='glm-4',
    api_key=os.getenv('zhipu_key'),
    base_url=os.getenv('zhipu_base_url'),
    temperature=0
)

In [20]:
#定义chain
from langchain_neo4j import GraphCypherQAChain
chain=GraphCypherQAChain.from_llm(graph=graph,llm=llm,verbose=True,allow_dangerous_requests=True)

In [21]:
result=chain.invoke({
    'query':'华为有哪些产品?'
})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: '华为'})<-[:Belong]-(s:SPU) RETURN s.name AS product;
Full Context:
[{'product': '华为智慧屏 4K全面屏智能电视机'}, {'product': '华为Mate 40 pro'}, {'product': '华为HUAWEI二手笔记本MateBook13触屏2K全面屏'}, {'product': 'HAWEIAI Book 2025英特尔14 Pro满血独显笔记本'}]

> Finished chain.


In [22]:
result=chain.invoke(
    {
        'query':'Apple有哪些产品'
    }
)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: 'Apple'})<-[:Belong]-(:SPU)-[:Have]->(:Tag)
RETURN t.name, SPU.name, Tag.name


CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Variable `SPU` not defined (line 2, column 16 (offset: 85))
"RETURN t.name, SPU.name, Tag.name"
                ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

result=chain.invoke(
    {
        'query':'苹果有哪些产品?'
    }
)

In [23]:
result=chain.invoke(
    {
        'query':'苹果有哪些产品?'
    }
)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: '苹果'})-[:Belong]->(s:SPU) RETURN s.name AS 产品名称
Full Context:
[]

> Finished chain.


In [24]:
result=chain.invoke(
    {
        'query':'红米有哪些产品?'
    }
)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: '红米'})<-[:Belong]-(:SPU)-[:Have]->(:SKU) RETURN DISTINCT SKU.name AS ProductName;


CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Variable `SKU` not defined (line 1, column 84 (offset: 83))
"MATCH (t:Trademark {name: '红米'})<-[:Belong]-(:SPU)-[:Have]->(:SKU) RETURN DISTINCT SKU.name AS ProductName;"
                                                                                    ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

In [25]:
result=chain.invoke({
    'query':'华为有哪些产品?'
})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: '华为'})<-[:Belong]-(:SPU)-[:Have]->(:Product) RETURN DISTINCT Product.name AS ProductName


CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Variable `Product` not defined (line 1, column 88 (offset: 87))
"MATCH (t:Trademark {name: '华为'})<-[:Belong]-(:SPU)-[:Have]->(:Product) RETURN DISTINCT Product.name AS ProductName"
                                                                                        ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

你提供的信息非常关键！现在情况很清晰了：

---

### ✅ 当前行为（正常）：
- LLM 生成的 Cypher 是：
  ```cypher
  MATCH (t:Trademark {name: '华为'})<-[:Belong]-(s:SPU) RETURN s.name AS product;
  ```
- 返回结果正常，说明 **你的图数据中，“产品”实际是以 `SPU` 节点的 `name` 属性存储的**。
- 并**没有用到 `Product` 节点**。

---

### ❓那为什么之前报错说 `Variable 'Product' not defined`？

> 很可能是因为 **LLM 的输出不稳定（随机性）**，在某次调用时生成了错误的 Cypher，比如：
> ```cypher
> MATCH (t:Trademark {name: '华为'})<-[:Belong]-(:SPU)-[:Have]->(:Product) RETURN Product.name
> ```
> 这种语句会触发你看到的错误。

而你现在运行的这次，LLM **恰好生成了正确的、简化的查询**（直接从 SPU 取名），所以成功了。

---

### 🤖 为什么 LLM 有时对、有时错？

1. **LLM 具有随机性**（即使 temperature=0，在某些实现中仍有微小波动）
2. **你的 schema 描述不明确**：LangChain 没有强制告诉 LLM “产品信息在 SPU 节点上”，所以它可能“脑补”出一个 `Product` 节点。
3. **缺少 schema 约束**：`GraphCypherQAChain` 默认只靠 LLM 自由发挥，容易幻觉。

---

### ✅ 解决方案：**固定查询逻辑 + 防止幻觉**

#### ✅ 方法 1：使用 `cypher_query_corrector`（推荐）

明确告诉 chain：**只有这些节点和关系是合法的**。

```python
from langchain_neo4j.chains.graph_qa.cypher_utils import CypherQueryCorrector, Schema

# 根据你的真实数据定义 schema
schema = Schema(
    nodes=["Trademark", "SPU"],  # 注意：没有 Product！
    relationships=[("Trademark", "Belong", "SPU")],  # 方向：Trademark <- Belong - SPU
)

corrector = CypherQueryCorrector(schema)

chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True,
    cypher_query_corrector=corrector  # 👈 关键！
)
```

这样即使 LLM 试图生成 `(:Product)`，corrector 也会将其过滤或修正为合法结构。

---

#### ✅ 方法 2：自定义 Prompt（更彻底）

你可以重写提示词，明确指示“产品名称存储在 SPU 节点的 name 属性中”。

```python
from langchain_core.prompts import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """
你是一个 Neo4j 专家。根据以下图数据库 schema 回答问题：

节点类型：
- Trademark: 品牌，属性包括 name
- SPU: 标准产品单元（代表具体商品），属性包括 name

关系：
- (Trademark)<-[:Belong]-(SPU)：表示该 SPU 属于某个品牌

注意：
- 所有“产品名称”都来自 SPU 节点的 name 属性
- 不要假设存在 Product 节点
- 只使用上述节点和关系
- 返回字段别名为 product

问题：{question}
"""

cypher_prompt = PromptTemplate.from_template(CYPHER_GENERATION_TEMPLATE)

chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    cypher_prompt=cypher_prompt,
    verbose=True,
    allow_dangerous_requests=True
)
```

这能**从根本上杜绝 LLM 幻觉出 `Product` 节点**。

---

### 🔍 验证你的数据模型

运行以下 Cypher，确认是否真的没有 `Product` 节点：
```cypher
MATCH (p:Product) RETURN count(p);  // 应该返回 0
MATCH (s:SPU) RETURN s.name LIMIT 5;  // 看看产品名是否在这里
```

如果确实没有 `Product` 节点，那就**不要在任何地方引用它**。

---

### ✅ 总结

| 问题 | 原因 | 解决 |
|------|------|------|
| 有时成功，有时报 `Product not defined` | LLM 随机生成了不存在的 `Product` 节点 | 1. 使用 `CypherQueryCorrector`2. 自定义 prompt 明确 schema3. 确保不引入不存在的节点 |

你现在能成功，是因为 LLM **碰巧生成了正确的查询**。为了**稳定可靠**，必须通过 schema 或 prompt 约束 LLM 的输出。

需要我帮你写完整的 `Schema` 定义或 prompt 模板吗？